In [ ]:
import torch
import re
import json
from torch.nn import CrossEntropyLoss
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from rapidfuzz import fuzz

In [ ]:
MODEL_PATH        = "/mnt/storage_C1/igorzwirtes/poster_ic/qwen2.5coder"
SPIDER_DB_DIR     = "/mnt/storage_C1/igorzwirtes/poster_ic/spider/database"  # para os .sqlite
SPIDER_TABLES     = "/mnt/storage_C1/igorzwirtes/poster_ic/spider/tables.json"  # para o schema
OUTPUT_DIR        = "./qwen-spider-finetuned/sft_com_schema"
USE_BF16          = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16          = torch.cuda.is_available() and not USE_BF16
USE_CPU           = not torch.cuda.is_available()

LR                = 1e-4
NUM_EPOCHS        = 3
GRAD_ACCUM        = 16
WARMUP_STEPS      = 50
MAX_NEW_TOKENS    = 156      
LOGGING_STEPS     = 50
EVAL_STEPS        = 200
SAVE_STEPS        = 200
SAVE_TOTAL_LIMIT  = 3
MAX_PROMPT_TOKENS = 500    
TARGET_MAX        = 350    

In [ ]:
def get_relevant_tables(question: str, db: dict, top_k: int = 8) -> set[int]:
    q = question.lower()
    scores = {}

    for i, table in enumerate(db["table_names_original"]):
        table_score = fuzz.partial_ratio(table.lower(), q)

        cols = [c[1].lower() for c in db["column_names_original"] if c[0] == i]
        col_score = max((fuzz.partial_ratio(c, q) for c in cols), default=0)

        scores[i] = max(table_score, col_score * 0.8)

    # ranking + filtro leve
    top = sorted(scores, key=scores.get, reverse=True)
    top = [i for i in top if scores[i] > 5][:top_k]

    seed = set(top)

    # FK expansion (1-hop)
    fk_tables = set(seed)

    for fk in db.get("foreign_keys", []):
        t1 = db["column_names_original"][fk[0]][0]
        t2 = db["column_names_original"][fk[1]][0]

        if t1 in seed or t2 in seed:
            fk_tables.add(t1)
            fk_tables.add(t2)

    return fk_tables

In [ ]:
with open(SPIDER_TABLES) as f:
    tables_data = json.load(f)

def format_schema_compact(db: dict, question: str, max_cols: int = 8) -> str:
    col_names = db["column_names_original"]
    col_types = db["column_types"]
    pks       = set(db.get("primary_keys", []))

    fk_map = {}
    for fk in db.get("foreign_keys", []):
        src, ref      = fk[0], fk[1]
        ref_table     = db["table_names_original"][col_names[ref][0]]
        ref_col       = col_names[ref][1]
        fk_map.setdefault(src, []).append(f"{ref_table}.{ref_col}")

    relevant_tables = get_relevant_tables(question, db)
    q_tokens        = set(re.sub(r"[^\w\s]", "", question.lower()).split())

    lines = []
    for i, table in enumerate(db["table_names_original"]):
        if i not in relevant_tables:
            continue

        cols_data = [
            (idx, col[1], col_types[idx])
            for idx, col in enumerate(col_names)
            if col[0] == i
        ]

        def col_score(item):
            idx, name, _ = item
            score = fuzz.partial_ratio(name.lower(), question.lower())
            if any(t in name.lower() for t in q_tokens): score += 20
            if idx in pks:    score += 30
            if idx in fk_map: score += 25
            return score

        cols_sorted = sorted(cols_data, key=col_score, reverse=True)[:max_cols]
        cols_sorted.sort(key=lambda x: x[0])

        col_parts = []
        for idx, name, ctype in cols_sorted:
            part = f"{name} {ctype}"
            if idx in pks:    part += " PK"
            if idx in fk_map: part += f" FK→{fk_map[idx]}"
            col_parts.append(part)

        lines.append(f"{table}({', '.join(col_parts)})")

    return "\n".join(lines)

tables_index = {db["db_id"]: db for db in tables_data}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

In [ ]:
dataset = load_dataset("xlangai/spider")

def format_example(example):
    db = tables_index[example["db_id"]]
    q  = example["question"]

    for max_cols in [8, 6, 4, 3]:
        schema = format_schema_compact(db, q, max_cols=max_cols)

        messages = [
            {"role": "system", "content": "Convert the question to a valid SQLite query. Output SQL only. Always use table aliases when joining multiple tables."},
            {"role": "user",   "content": f"Schema ({example['db_id']}):\n{schema}\n\nQuestion: {q}"},
            {"role": "assistant", "content": example["query"]},
        ]

        prompt   = tokenizer.apply_chat_template(messages, tokenize=False)
        n_tokens = len(tokenizer.encode(prompt))

        if n_tokens <= TARGET_MAX:
            break

    # hard cap: trunca linhas do schema até caber
    if n_tokens > MAX_PROMPT_TOKENS:
        schema_lines = schema.split("\n")
        while len(schema_lines) > 1 and n_tokens > MAX_PROMPT_TOKENS:
            schema_lines.pop()
            messages[1]["content"] = f"Schema ({example['db_id']}):\n{chr(10).join(schema_lines)}\n\nQuestion: {q}"
            prompt   = tokenizer.apply_chat_template(messages, tokenize=False)
            n_tokens = len(tokenizer.encode(prompt))

    return {"text": prompt}

train_data = dataset["train"].map(format_example)
val_data   = dataset["validation"].map(format_example)

In [ ]:
lengths = [len(tokenizer(x["text"])["input_ids"]) for x in train_data]
lengths.sort()
p95 = lengths[int(len(lengths) * 0.99)]
print(f"p99: {p95} tokens") 

In [ ]:
class SQLOnlyTrainer(SFTTrainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._assistant_prefix_ids = tokenizer.encode(
            "<|im_start|>assistant\n", add_special_tokens=False
        )

    def _find_assistant_start(self, ids: list[int]) -> int:
        prefix = self._assistant_prefix_ids
        n = len(prefix)
        for j in range(len(ids) - n, -1, -1):
            if ids[j:j + n] == prefix:
                return j + n
        return len(ids)  # fallback seguro: mascara tudo

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        input_ids      = inputs["input_ids"]
        attention_mask = inputs["attention_mask"]
        labels         = input_ids.clone()

        for i in range(labels.shape[0]):
            start = self._find_assistant_start(input_ids[i].tolist())
            labels[i, :start] = -100

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        return (outputs.loss, outputs) if return_outputs else outputs.loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)

        with torch.no_grad():
            input_ids      = inputs["input_ids"]
            attention_mask = inputs["attention_mask"]
            labels         = input_ids.clone()

            for i in range(labels.shape[0]):
                start = self._find_assistant_start(input_ids[i].tolist())
                labels[i, :start] = -100

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

        return (outputs.loss.detach(), None, None)

In [ ]:
compute_dtype = torch.bfloat16 if USE_BF16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=compute_dtype,
)
model.config.use_cache = False
model.config.pad_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.eos_token_id
model.generation_config.bos_token_id = tokenizer.bos_token_id

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

sft_config = SFTConfig(
    output_dir="./qwen-spider-finetuned",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_32bit",
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    bf16=USE_BF16,
    fp16=USE_FP16,
    use_cpu=USE_CPU,
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=False,
    report_to="none",
    max_length=MAX_PROMPT_TOKENS + MAX_NEW_TOKENS,
    dataset_text_field="text",
    disable_tqdm=False,
    logging_strategy="steps",
    pad_token="<|endoftext|>",
)

trainer = SQLOnlyTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=val_data,
    peft_config=lora_config,
    args=sft_config,
)

In [ ]:
trainer.train()
trainer.save_model(OUTPUT_DIR)